In [10]:
!git clone https://github.com/directedbyshawn/ML-FINAL-PROJECT.git

Cloning into 'ML-FINAL-PROJECT'...
remote: Enumerating objects: 8023, done.
remote: Counting objects: 100% (8023/8023), done.
remote: Compressing objects: 100% (7982/7982), done.
remote: Total 8023 (delta 32), reused 8007 (delta 21), pack-reused 0
Receiving objects: 100% (8023/8023), 34.45 MiB | 5.51 MiB/s, done.
Resolving deltas: 100% (32/32), done.


In [43]:
import numpy as np
import matplotlib.pyplot as plt
import os
import imageio
import glob
import cv2
from sklearn.model_selection import train_test_split
from keras.utils.np_utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense
from keras.optimizers import Adam
from keras.layers import Dropout, Flatten
from keras.layers.convolutional import Conv2D, MaxPooling2D
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import confusion_matrix



In [13]:
def divide_data(train, test):
    # 60, 20, 20
    # set aside 20% of train and test data for evaluation
    X_train, X_test, y_train, y_test = train_test_split(
        train, test, test_size=0.2, shuffle=True, random_state=42)

    # Use the same function above for the validation set
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_train, y_train, test_size=0.25, shuffle=True, random_state=42)  # 0.25 x 0.8 = 0.2

    return X_train, X_valid, X_test, y_train, y_valid, y_test


def load_data():
    train = []
    test = []
    
    for n_class in range(0, 43):

        path = os.getcwd() + "/data/" + str(n_class) + "/"

        class_train = []
        
        for im_path in glob.glob(path + "*.png"):
            image = imageio.v2.imread(im_path)

            class_train.append(image)

        # to "normalize" the number of each class
        if len(class_train) > 150:
            np.random.shuffle(class_train)
            class_train = class_train[:150][:][:]

        train.extend(class_train)
        test.extend([n_class] * len(class_train))


    train = np.array(train)
    test = np.array(test)

    X_train, X_valid, X_test, y_train, y_valid, y_test = divide_data(train, test)

    return X_train, X_valid, X_test, y_train, y_valid, y_test

In [29]:
X_train, X_valid, X_test, y_train, y_valid, y_test = load_data()
print(X_train.shape, X_valid.shape, X_test.shape,y_train.shape, y_valid.shape, y_test.shape)

(1914, 32, 32, 3) (638, 32, 32, 3) (638, 32, 32, 3) (1914,) (638,) (638,)


In [24]:
def grayscale(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return image


def equalize(image):
    image = cv2.equalizeHist(image)
    return image


def preprocessing(image):
    image = grayscale(image)
    image = equalize(image)
    image = image/255
    return image

In [30]:
X_train = np.array(list(map(preprocessing, X_train)))
X_valid = np.array(list(map(preprocessing, X_valid)))
X_test = np.array(list(map(preprocessing, X_test)))

X_train = X_train.reshape(X_train.shape[0], 32, 32, 1)
X_valid = X_valid.reshape(X_valid.shape[0], 32, 32, 1)
X_test = X_test.reshape(X_test.shape[0], 32, 32, 1)

y_train = to_categorical(y_train, 43)
y_valid = to_categorical(y_valid, 43)
y_test = to_categorical(y_test, 43)

In [31]:
print(X_train.shape, X_valid.shape, X_test.shape,y_train.shape, y_valid.shape, y_test.shape)

(1914, 32, 32, 1) (638, 32, 32, 1) (638, 32, 32, 1) (1914, 43) (638, 43) (638, 43)


In [33]:
def compile_model():
    model = Sequential()
    model.add(Conv2D(60, (5, 5), input_shape=(32, 32, 1), activation='relu'))
    model.add(Conv2D(60, (5, 5), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Conv2D(30, (3, 3), activation='relu'))
    model.add(Conv2D(30, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Flatten())
    model.add(Dense(500, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(43, activation='softmax'))

    # Compile Model
    model.compile(Adam(learning_rate=0.001),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [34]:
model = compile_model()
print(model.summary())

2022-10-28 14:44:37.237832: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/dmytro/.local/lib/python3.8/site-packages/cv2/../../lib64:
2022-10-28 14:44:37.238401: W tensorflow/stream_executor/cuda/cuda_driver.cc:263] failed call to cuInit: UNKNOWN ERROR (303)
2022-10-28 14:44:37.238453: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (dmytro-Lenovo-V330-14ARR): /proc/driver/nvidia/version does not exist
2022-10-28 14:44:37.239961: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 28, 28, 60)        1560      
                                                                 
 conv2d_1 (Conv2D)           (None, 24, 24, 60)        90060     
                                                                 
 max_pooling2d (MaxPooling2D  (None, 12, 12, 60)       0         
 )                                                               
                                                                 
 conv2d_2 (Conv2D)           (None, 10, 10, 30)        16230     
                                                                 
 conv2d_3 (Conv2D)           (None, 8, 8, 30)          8130      
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 4, 4, 30)         0         
 2D)                                                    

In [35]:
history = model.fit(X_train, y_train, validation_data=(X_valid, y_valid), batch_size=32, epochs=10)
print(history)

Epoch 1/10
60/60 [==============================] - 18s 285ms/step - loss: 2.8948 - accuracy: 0.2508 - val_loss: 1.2771 - val_accuracy: 0.6614
Epoch 2/10
60/60 [==============================] - 16s 265ms/step - loss: 0.8320 - accuracy: 0.7696 - val_loss: 0.4481 - val_accuracy: 0.8903
Epoch 3/10
60/60 [==============================] - 16s 273ms/step - loss: 0.3782 - accuracy: 0.8939 - val_loss: 0.3386 - val_accuracy: 0.9279
Epoch 4/10
60/60 [==============================] - 22s 365ms/step - loss: 0.2545 - accuracy: 0.9274 - val_loss: 0.2854 - val_accuracy: 0.9107
Epoch 5/10
60/60 [==============================] - 22s 374ms/step - loss: 0.1867 - accuracy: 0.9446 - val_loss: 0.2569 - val_accuracy: 0.9295
Epoch 6/10
60/60 [==============================] - 23s 389ms/step - loss: 0.1047 - accuracy: 0.9707 - val_loss: 0.2235 - val_accuracy: 0.9483
Epoch 7/10
60/60 [==============================] - 22s 364ms/step - loss: 0.0877 - accuracy: 0.9697 - val_loss: 0.3587 - val_accuracy: 0.9248

In [36]:
predict_x = model.predict(X_test)

20/20 [==============================] - 1s 47ms/step


In [37]:
y_pred = []
for y in predict_x:
    pred = np.argmax(y)
    y_pred.append(pred)
    
y_my_test = []
for y in y_test:
    pred = np.argmax(y)
    y_my_test.append(pred)

y_pred = np.array(y_pred)
y_test = np.array(y_my_test)

In [40]:
precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)

(0.9439780866846169, 0.9382182813654204, 0.9374034455961356, None)

In [44]:
confusion_matrix(y_test, y_pred)

array([[36,  0,  0, ...,  0,  0,  0],
       [ 0, 23,  1, ...,  0,  1,  0],
       [ 0,  2, 32, ...,  0,  0,  0],
       ...,
       [ 0,  0,  0, ..., 19,  0,  0],
       [ 0,  0,  0, ...,  0,  8,  0],
       [ 0,  0,  0, ...,  0,  0,  3]])